# 🔗 HPVD × Knowledge Layer — Integration Demo

Notebook ini mendemonstrasikan alur **end-to-end** HPVD mengambil data dari Knowledge Layer (KL) yang live, 
menjalankan pipeline retrieval, dan menghasilkan **J14 (RetrievalRaw)**, **J15 (PhaseFilteredSet)**, dan **J16 (AnalogFamilyAssignment)**.

---

**Pipeline:**
```
KL API (live) → KLDocumentLoader → DocumentChunk[] → HPVD Pipeline → J14 → J15 → J16
```

## 1. Connect to Knowledge Layer

In [ ]:
from src.hpvd.adapters.kl_client import KLClient

KL_URL = "https://knowledge-layer-production.up.railway.app"
client = KLClient(base_url=KL_URL)

# Health check
health = client.health_check()
print(f"✅ KL API reachable: {health}")

## 2. Fetch Documents from KL

In [ ]:
TENANT_ID = "BANKING_CORE"

documents = client.list_documents(TENANT_ID, limit=100)
print(f"📄 Total documents in KL for tenant '{TENANT_ID}': {len(documents)}")
print()

# Show summary by document type
from collections import Counter
type_counts = Counter(d.document_type for d in documents)
print("Document types:")
for doc_type, count in type_counts.most_common():
    print(f"  {doc_type:25s} → {count} docs")

In [ ]:
# Show first 10 documents as a table
import pandas as pd

df_docs = pd.DataFrame([
    {
        "id": d.id[:8] + "...",
        "title": d.title[:60],
        "type": d.document_type,
        "created_at": d.created_at[:19] if d.created_at else "",
    }
    for d in documents[:10]
])
df_docs

## 3. Load & Convert to HPVD DocumentChunks

In [ ]:
from src.hpvd.adapters.kl_document_loader import KLDocumentLoader

loader = KLDocumentLoader(client)
chunks = loader.load_as_chunks(TENANT_ID)

print(f"🧩 Converted {len(chunks)} documents → {len(chunks)} DocumentChunks")
print()

# Show topic distribution
topic_counts = Counter(c.topic for c in chunks)
print("Topic distribution (mapped from document_type):")
for topic, count in topic_counts.most_common():
    print(f"  {topic:25s} → {count} chunks")

In [ ]:
# Show sample chunks
df_chunks = pd.DataFrame([
    {
        "chunk_id": c.chunk_id[:15] + "...",
        "topic": c.topic,
        "doc_type": c.doc_type,
        "text_preview": c.text[:80],
    }
    for c in chunks[:10]
])
df_chunks

## 4. Build HPVD Index & Run Pipeline

Kita akan menjalankan pipeline HPVD lengkap:
1. **Build index** dari DocumentChunks (embedding + FAISS)
2. **Construct J13** (PostCoreQuery) — simulasi query dari Serving Adapter
3. **Run pipeline** → produce J14, J15, J16

In [ ]:
from src.hpvd.adapters.strategies.document_strategy import (
    DocumentRetrievalStrategy, 
    DocumentRetrievalConfig,
)
from src.hpvd.adapters.pipeline_engine import HPVDPipelineEngine

# Build index
strategy = DocumentRetrievalStrategy(
    DocumentRetrievalConfig(min_similarity=0.0)
)
strategy.build_index(chunks)
print(f"✅ FAISS index built with {len(chunks)} vectors")

# Create pipeline
pipeline = HPVDPipelineEngine(strategies=[strategy])
print(f"✅ Pipeline ready (registered domains: document)")

### 4.1 Construct J13 — PostCoreQuery

Ini adalah input query yang dalam pipeline penuh akan datang dari **Serving Adapter (Stage 11)**.

In [ ]:
import json

j13_query = {
    "query_id": "Q_LOAN_SUPPORT_001",
    "scope": {
        "domain": "banking",
        "action_class": "LOAN_EXECUTION",
    },
    "allowed_topics": [],  # no topic filter = search all
    "allowed_doc_types": [],
    "query_payload": {
        "text": "credit risk assessment loan default payment",
    },
}

print("📥 J13 — PostCoreQuery:")
print(json.dumps(j13_query, indent=2))

### 4.2 Execute Pipeline → J14, J15, J16

In [ ]:
# Run the pipeline
output = pipeline.process_query(j13_query, k=10)

print(f"✅ Pipeline complete!")
print(f"   J14 candidates : {len(output.j14.candidates)}")
print(f"   J15 accepted   : {len(output.j15.accepted)}")
print(f"   J15 rejected   : {len(output.j15.rejected)}")
print(f"   J16 families   : {output.j16.total_families}")
print(f"   J16 members    : {output.j16.total_members}")

## 5. Inspect J14 — HPVD Retrieval Raw

Hasil retrieval mentah: daftar kandidat analog dengan **calibrated similarity score**.

In [ ]:
print("📤 J14 — HPVD_RetrievalRaw")
print(f"   schema_id : {output.j14.schema_id}")
print(f"   query_id  : {output.j14.query_id}")
print(f"   domain    : {output.j14.domain}")
print(f"   candidates: {len(output.j14.candidates)}")
print()

# Show candidates as table
df_j14 = pd.DataFrame([
    {
        "rank": i + 1,
        "candidate_id": c["candidate_id"][:20] + "...",
        "score": round(c["score"], 4),
        "topic": c.get("metadata", {}).get("topic", ""),
        "doc_type": c.get("metadata", {}).get("doc_type", ""),
        "text_preview": c.get("metadata", {}).get("text_preview", "")[:50],
    }
    for i, c in enumerate(output.j14.candidates)
])
df_j14

In [ ]:
# Full J14 JSON
print(json.dumps(output.j14.to_dict(), indent=2, ensure_ascii=False))

## 6. Inspect J15 — Phase Filtered Set

Kandidat yang sudah difilter berdasarkan **phase/topic consistency**.

In [ ]:
print("📤 J15 — PhaseFilteredSet")
print(f"   schema_id      : {output.j15.schema_id}")
print(f"   accepted count : {len(output.j15.accepted)}")
print(f"   rejected count : {len(output.j15.rejected)}")
print(f"   filter_criteria: {output.j15.filter_criteria}")
print()

# Show accepted candidates
if output.j15.accepted:
    df_j15 = pd.DataFrame([
        {
            "candidate_id": c["candidate_id"][:20] + "...",
            "score": round(c["score"], 4),
            "status": "ACCEPTED",
        }
        for c in output.j15.accepted
    ])
    display(df_j15)

if output.j15.rejected:
    print(f"\nRejected ({len(output.j15.rejected)} candidates):")
    for r in output.j15.rejected:
        print(f"  {r['candidate_id'][:20]}... — reason: {r.get('reason', 'N/A')}")

In [ ]:
# Full J15 JSON
print(json.dumps(output.j15.to_dict(), indent=2, ensure_ascii=False))

## 7. Inspect J16 — Analog Family Assignment

Kandidat yang sudah dikelompokkan menjadi **analog families** dengan:
- `coherence` — seberapa koheren family tersebut
- `structural_signature` — ciri struktural family
- `uncertainty_flags` — apakah ada ketidakpastian

In [ ]:
print("📤 J16 — AnalogFamilyAssignment")
print(f"   schema_id     : {output.j16.schema_id}")
print(f"   total_families: {output.j16.total_families}")
print(f"   total_members : {output.j16.total_members}")
print()

for fam in output.j16.families:
    print(f"{'='*60}")
    print(f"Family: {fam['family_id']}")
    print(f"  Phase/Topic    : {fam['structural_signature']['phase']}")
    print(f"  Members        : {fam['coherence']['size']}")
    print(f"  Mean confidence: {fam['coherence']['mean_confidence']:.4f}")
    print(f"  Dispersion     : {fam['coherence']['dispersion']:.4f}")
    print(f"  Weak support   : {fam['uncertainty_flags']['weak_support']}")
    print(f"  Phase boundary : {fam['uncertainty_flags']['phase_boundary']}")
    print(f"  Members:")
    for m in fam['members']:
        print(f"    - {m['candidate_id'][:25]}...  score={m['score']:.4f}")

In [ ]:
# Family summary table
df_j16 = pd.DataFrame([
    {
        "family_id": f["family_id"],
        "topic": f["structural_signature"]["phase"],
        "members": f["coherence"]["size"],
        "mean_confidence": round(f["coherence"]["mean_confidence"], 4),
        "dispersion": round(f["coherence"]["dispersion"], 4),
        "weak_support": f["uncertainty_flags"]["weak_support"],
    }
    for f in output.j16.families
])
df_j16

In [ ]:
# Full J16 JSON
print(json.dumps(output.j16.to_dict(), indent=2, ensure_ascii=False))

## 8. Save Results to File

Simpan output lengkap ke JSON file untuk referensi.

In [ ]:
import os
from datetime import datetime

# Save full pipeline output
output_dir = "hpvd_outputs/kl_integration"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(output_dir, f"pipeline_output_{timestamp}.json")

full_output = {
    "metadata": {
        "timestamp": timestamp,
        "kl_url": KL_URL,
        "tenant_id": TENANT_ID,
        "total_kl_documents": len(documents),
        "total_chunks": len(chunks),
    },
    "j13_query": j13_query,
    "pipeline_output": output.to_dict(),
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(full_output, f, indent=2, ensure_ascii=False)

print(f"💾 Output saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file):,} bytes")

## 9. Try Different Queries

Coba ubah query di bawah untuk lihat bagaimana HPVD meresponse query yang berbeda.

In [ ]:
# Try different queries!
queries = [
    {"text": "bank deliberation approval decision", "label": "Bank Decision"},
    {"text": "legal notice payment intimation", "label": "Legal Notice"},
    {"text": "fidejussione garanzia guarantee", "label": "Guarantee"},
    {"text": "loan contract finanziamento", "label": "Contract"},
]

results_summary = []
for q in queries:
    j13 = {
        "query_id": f"multi_q_{q['label'].replace(' ', '_').lower()}",
        "scope": {"domain": "banking"},
        "allowed_topics": [],
        "query_payload": {"text": q["text"]},
    }
    out = pipeline.process_query(j13, k=5)
    
    top_candidate = out.j14.candidates[0] if out.j14.candidates else {}
    results_summary.append({
        "query": q["label"],
        "candidates": len(out.j14.candidates),
        "families": out.j16.total_families,
        "top_score": round(top_candidate.get("score", 0), 4),
        "top_topic": top_candidate.get("metadata", {}).get("topic", ""),
    })

df_results = pd.DataFrame(results_summary)
df_results

## 10. Topic-Filtered Query

Coba query dengan `allowed_topics` untuk membatasi retrieval ke topic tertentu.

In [ ]:
# Query with topic filter
j13_filtered = {
    "query_id": "Q_FILTERED_CREDIT",
    "scope": {"domain": "banking"},
    "allowed_topics": ["credit_analysis"],  # Only credit-related docs
    "query_payload": {"text": "credit risk assessment"},
}

out_filtered = pipeline.process_query(j13_filtered, k=10)

print(f"🔍 Filtered query (topic=credit_analysis):")
print(f"   Candidates: {len(out_filtered.j14.candidates)}")
print(f"   Families  : {out_filtered.j16.total_families}")
print()

for c in out_filtered.j14.candidates:
    print(f"  score={c['score']:.4f}  topic={c['metadata']['topic']}  {c['metadata'].get('text_preview', '')[:50]}")

---

## Summary

| Stage | Output | Description |
|-------|--------|-------------|
| KL API | Documents | 36 banking PDFs from 6 cases |
| Loader | DocumentChunks | Converted with topic mapping |
| J14 | RetrievalRaw | Ranked candidates with similarity scores |
| J15 | PhaseFilteredSet | Accepted/rejected by phase filter |
| J16 | AnalogFamilyAssignment | Grouped into families with coherence metrics |